In [1]:
from types import SimpleNamespace
from h5_utils import save, load

In [2]:
RAW_DVL = load('1_RAW_DATA.h5', 'RAW_DVL')

Loaded RAW_DVL from 1_RAW_DATA.h5


<div style="font-family: 'Times New Roman', Georgia, serif; background-color: #fefdf8; padding: 20px; border: 2px solid #d4c4a8; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); line-height: 1.6; max-width: 100%; box-sizing: border-box; overflow: hidden;">

<h3 style="color: #5d4e37; margin: 25px 0 15px 0; width: 100%; max-width: 100%; word-wrap: break-word; overflow-wrap: break-word; hyphens: auto; box-sizing: border-box; font-weight: 800; font-size: 20px;">We clean up the DVL speed readings: any jump over 2 m/s is most likely a sensor glitch. So we look for those spikes, count how many there are, and replace each spike with the last good value to smooth the data.</h3>

</div>

In [3]:
Threshold_DVL = 2 # The diver with the robot cannot "run" with more than 2 m/s underwater 
DVL_FILTERED = SimpleNamespace(**{field: getattr(RAW_DVL, field).copy() for field in vars(RAW_DVL)}) # Coping RAW_DVL to DVL_FILTERED so we can overwrite outliers in place
nbr_outliers = 0

# Loop through each sample
for i in range(1, len(RAW_DVL.U)):
    if (abs(RAW_DVL.U[i]) > Threshold_DVL or abs(RAW_DVL.V[i]) > Threshold_DVL or  abs(RAW_DVL.W[i]) > Threshold_DVL): ## If any velocity component exceeds the threshold, it’s an outlier
        nbr_outliers += 1
        DVL_FILTERED.U[i] = DVL_FILTERED.U[i-1]   # \
        DVL_FILTERED.V[i] = DVL_FILTERED.V[i-1]   #  Replace the bad sample with the previous one
        DVL_FILTERED.W[i] = DVL_FILTERED.W[i-1]   # /

print('Number of outliers:', nbr_outliers, 'out of ', len(RAW_DVL.U))

Number of outliers: 802 out of  20538


In [4]:
save('HDF5_DATA/2_FILTERED_DVL.h5', 'DVL_FILTERED', DVL_FILTERED, mode='w')

Saved to HDF5_DATA/2_FILTERED_DVL.h5 -> DVL_FILTERED
